# Išlaidų reikalavimo analizė

Šiame užrašų knygyne demonstruojama, kaip sukurti agentus, kurie naudoja papildinius vietinių kvitų nuotraukoms apdoroti, sudaryti išlaidų reikalavimo el. laišką ir vizualizuoti išlaidų duomenis naudojant pyragų diagramą. Agentai dinamiškai parenka funkcijas pagal užduoties kontekstą.

Žingsniai:
1. OCR agentas apdoroja vietinę kvito nuotrauką ir ištraukia kelionių išlaidų duomenis.
2. El. pašto agentas sugeneruoja išlaidų reikalavimo el. laišką.

### Kelionių išlaidų scenarijaus pavyzdys:
Įsivaizduokite, kad esate darbuotojas, keliaujantis į verslo susitikimą kitame mieste. Jūsų įmonės politika numato kompensuoti visas pagrįstas su kelionėmis susijusias išlaidas. Štai galimų kelionių išlaidų sąrašas:
- Transportas:
Lėktuvo bilietai į abi puses iš jūsų namų miesto į kelionės miestą.
Taksi arba automobilio nuomos paslaugos iš ir į oro uostą.
Vietinis transportas kelionės mieste (pvz., viešasis transportas, automobilių nuoma ar taksi).

- Apgyvendinimas:
Viešbučio apgyvendinimas trims naktims verslo klasės vidutinės kainos viešbutyje netoli susitikimo vietos.

- Maistas:
Kasdienė maisto išlaidų norma pusryčiams, pietums ir vakarienei remiantis įmonės dienpinigiais.

- Kitos išlaidos:
Parkavimo mokesčiai oro uoste.
Interneto ryšio mokesčiai viešbutyje.
Arbatpinigiai arba mažos paslaugų mokesčiai.

- Dokumentacija:
Jūs pateikiate visus kvitus (skrydžiams, taksi, viešbučiui, maistui ir kt.) bei užpildytą išlaidų ataskaitą kompensacijai gauti.


## Importuoti reikalingas bibliotekas

Importuokite būtinas bibliotekas ir modulius užrašinei.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import Content, Message, tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Apibrėžkite išlaidų modelius

 Sukurkite Pydantic modelį individualioms išlaidoms ir ExpenseFormatter klasę, kuri konvertuos vartotojo užklausą į struktūrizuotus išlaidų duomenis.

 Kiekviena išlaida bus pateikta formatu:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Įrankių apibrėžimas – El. pašto generavimas

Sukurkite įrankio funkciją, skirtą sugeneruoti el. laišką išlaidų ataskaitai pateikti.
- Šis įrankis naudoja `@tool` dekoratorių iš Microsoft Agent Framework.
- Jis apskaičiuoja bendrą išlaidų sumą ir formatuoja duomenis į el. laiško tekstą.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Kelionės išlaidų kvitų vaizdų įkėlimas

Sukurkite pagalbinę funkciją, skirtą įkelti kvitų vaizdus kaip natūralų multimodalinį turinį.
- Ji skaito kvito vaizdo baitus ir apgaubia juos Agent Framework `Content`, kad agentas galėtų juos analizuoti.


In [ ]:
def load_receipt_image(image_path: str = "receipt.jpg") -> Content:
    """Load a receipt image as native multimodal content."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    return Content.from_data(image_bytes, "image/jpeg")

## Tvarkymo išlaidos

Apibrėžkite agentus ir sukonstruokite juos į seką naudojant `WorkflowBuilder`.
- OCR agentas išgauna struktūrizuotus išlaidų duomenis iš čekio paveikslėlio, pateikto vartotojo žinutėje.
- El. pašto agentas paima išgautus duomenis ir sukuria profesionalią išlaidų pareiškimo el. laišką naudodamas `generate_expense_email` įrankį.
- `WorkflowBuilder` su `add_edge` sukuria seką: OCR agentas → El. pašto agentas.


In [ ]:
ocr_agent = client.as_agent(
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Analyze the receipt image supplied in the user message and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Pagrindinė funkcija

Sukurkite nuoseklų darbo eigą ir paleiskite ją, kad apdorotumėte čekio vaizdą ir sugeneruotumėte išlaidų deklaracijos el. laišką.


> **Pastaba:** Gautų baitų žinutės yra įvyniotos natūralioje Agent Framework multimodalinėje `Content`, todėl Foundry juos gauna kaip vaizdo įvestį, o ne įprastą tekstą.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
receipt_message = Message(
    role="user",
    contents=[prompt, load_receipt_image("receipt.jpg")],
)

events = workflow.run(
    receipt_message,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
